# Reproducible Analysis for Agree, Disagree, Explain

This notebook reproduces the public-repository analyses from `annotations_varierr.jsonl` and `annotations_livenli.jsonl`. It uses paths relative to the repository root and writes derived tables and figures to `outputs/`.

Both public JSONL files include persistent annotator identifiers: VariErr uses `annotator_ids`, and LiveNLI uses `worker_ids`. The annotator-level sections below therefore run directly on the released files.


## Setup


In [ ]:
import json
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

ROOT = Path.cwd()
DATASETS = {
    "VariErr": ROOT / "annotations_varierr.jsonl",
    "LiveNLI": ROOT / "annotations_livenli.jsonl",
}
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

LABEL_ORDER = ["entailment", "neutral", "contradiction"]
CATEGORY_ORDER = [
    "Coreference",
    "Syntactic",
    "Semantic",
    "Pragmatic",
    "Absence of Mention",
    "Logic Conflict",
    "Factual Knowledge",
    "Inferential Knowledge",
]
LABEL_COLORS = {
    "entailment": "#66c2a5",
    "neutral": "#fc8d62",
    "contradiction": "#8da0cb",
}

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
})


## Loading and Normalization


In [ ]:
def canon_label(label):
    value = (label or "").strip().lower()
    if value in {"e", "entailment", "true"}:
        return "entailment"
    if value in {"n", "neutral", "either"}:
        return "neutral"
    if value in {"c", "contradiction", "false"}:
        return "contradiction"
    return value or "<Unspecified>"


def canon_categories(value):
    if value is None:
        return ["<Unspecified>"]
    if isinstance(value, str):
        value = value.strip()
        return [value] if value else ["<Unspecified>"]
    if isinstance(value, list):
        cleaned = [str(item).strip() for item in value if str(item).strip()]
        return cleaned or ["<Unspecified>"]
    return [str(value).strip() or "<Unspecified>"]


def get_annotator_ids(row):
    ids = row.get("annotator_ids")
    if ids is None:
        ids = row.get("worker_ids")
    if ids is None:
        ids = row.get("worker_id")
    if ids is None:
        return []
    if not isinstance(ids, list):
        ids = [ids]
    return [str(item).strip() for item in ids if str(item).strip()]


def load_jsonl(path, dataset_name):
    rows = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            annotator_ids = get_annotator_ids(item)
            rows.append({
                "dataset": dataset_name,
                "line_no": line_no,
                "pairID": str(item.get("pairID") or item.get("id") or f"line_{line_no}"),
                "premise": item.get("premise", ""),
                "hypothesis": item.get("hypothesis", ""),
                "label": canon_label(item.get("gold_label")),
                "categories": canon_categories(item.get("explanation_category")),
                "explanation": item.get("explanation", ""),
                "note": item.get("note", ""),
                "annotator_ids": annotator_ids,
            })
    return pd.DataFrame(rows)


datasets = {name: load_jsonl(path, name) for name, path in DATASETS.items()}
all_rows = pd.concat(datasets.values(), ignore_index=True)

for name, frame in datasets.items():
    print(f"{name}: {len(frame):,} annotations, {frame['pairID'].nunique():,} unique pairs")
    print(f"  annotator identifiers present: {frame['annotator_ids'].map(bool).any()}")


## Dataset Overview


In [ ]:
def word_count(text):
    return len(str(text).strip().split()) if str(text).strip() else 0


overview_rows = []
for name, frame in datasets.items():
    category_tags = sum(len(cats) for cats in frame["categories"])
    overview_rows.append({
        "dataset": name,
        "annotations": len(frame),
        "unique_pairs": frame["pairID"].nunique(),
        "label_types": frame["label"].nunique(),
        "category_tags": category_tags,
        "avg_explanation_words": round(frame["explanation"].map(word_count).mean(), 2),
        "has_annotator_ids": frame["annotator_ids"].map(bool).any(),
    })

overview = pd.DataFrame(overview_rows)
overview.to_csv(OUTPUT_DIR / "dataset_overview.csv", index=False)
overview


## Label Distributions


In [ ]:
def label_distribution(frame):
    counts = frame["label"].value_counts().reindex(LABEL_ORDER, fill_value=0)
    total = counts.sum()
    return pd.DataFrame({
        "label": counts.index,
        "count": counts.values,
        "proportion": counts.values / total if total else 0,
    })


label_tables = []
fig, axes = plt.subplots(1, len(datasets), figsize=(11, 4), sharey=True)
if len(datasets) == 1:
    axes = [axes]

for ax, (name, frame) in zip(axes, datasets.items()):
    table = label_distribution(frame)
    table.insert(0, "dataset", name)
    label_tables.append(table)
    ax.bar(table["label"], table["proportion"], color=[LABEL_COLORS[label] for label in table["label"]], edgecolor="black")
    ax.set_title(name)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.0%}"))
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.5)

axes[0].set_ylabel("Proportion of annotations")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "label_distribution.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "label_distribution.pdf", bbox_inches="tight")
plt.show()

label_summary = pd.concat(label_tables, ignore_index=True)
label_summary.to_csv(OUTPUT_DIR / "label_distribution.csv", index=False)
label_summary


## Explanation Category Distributions


In [ ]:
def explode_categories(frame):
    return frame.explode("categories").rename(columns={"categories": "category"})


def ordered_categories(categories):
    present = set(categories)
    ordered = [cat for cat in CATEGORY_ORDER if cat in present]
    ordered.extend(sorted(cat for cat in present if cat not in CATEGORY_ORDER))
    return ordered


category_tables = []
fig, axes = plt.subplots(1, len(datasets), figsize=(13, 5), sharey=True)
if len(datasets) == 1:
    axes = [axes]

for ax, (name, frame) in zip(axes, datasets.items()):
    exploded = explode_categories(frame)
    categories = ordered_categories(exploded["category"].unique())
    counts = exploded["category"].value_counts().reindex(categories, fill_value=0)
    proportions = counts / counts.sum()
    table = pd.DataFrame({"dataset": name, "category": categories, "count": counts.values, "proportion": proportions.values})
    category_tables.append(table)

    ax.barh(categories, proportions.values, color=plt.cm.Set3(np.linspace(0, 1, len(categories))), edgecolor="black")
    ax.set_title(name)
    ax.set_xlim(0, max(0.01, proportions.max() * 1.15))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.0%}"))
    ax.grid(axis="x", linestyle="--", linewidth=0.5, alpha=0.5)

axes[0].set_ylabel("Explanation category")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "category_distribution.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "category_distribution.pdf", bbox_inches="tight")
plt.show()

category_summary = pd.concat(category_tables, ignore_index=True)
category_summary.to_csv(OUTPUT_DIR / "category_distribution.csv", index=False)
category_summary


## Label Distribution by Explanation Category


In [ ]:
def category_label_table(frame):
    exploded = explode_categories(frame)
    counts = exploded.groupby(["category", "label"]).size().unstack(fill_value=0)
    counts = counts.reindex(columns=LABEL_ORDER, fill_value=0)
    counts = counts.reindex(ordered_categories(counts.index))
    proportions = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    return counts, proportions


category_label_summaries = []
for name, frame in datasets.items():
    counts, proportions = category_label_table(frame)
    counts_out = counts.reset_index().rename(columns={"index": "category"})
    counts_out.insert(0, "dataset", name)
    counts_out.to_csv(OUTPUT_DIR / f"{name.lower()}_category_label_counts.csv", index=False)

    prop_out = proportions.reset_index().rename(columns={"index": "category"})
    prop_out.insert(0, "dataset", name)
    prop_out.to_csv(OUTPUT_DIR / f"{name.lower()}_category_label_proportions.csv", index=False)
    category_label_summaries.append(prop_out)

    ax = proportions.plot(
        kind="barh",
        stacked=True,
        figsize=(9, 5),
        color=[LABEL_COLORS[label] for label in LABEL_ORDER],
        edgecolor="black",
    )
    ax.set_title(name)
    ax.set_xlabel("Label proportion within category")
    ax.set_ylabel("Explanation category")
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.0%}"))
    ax.legend(title="NLI label", loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
    ax.grid(axis="x", linestyle="--", linewidth=0.5, alpha=0.5)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{name.lower()}_category_label_distribution.png", dpi=300, bbox_inches="tight")
    plt.savefig(OUTPUT_DIR / f"{name.lower()}_category_label_distribution.pdf", bbox_inches="tight")
    plt.show()

pd.concat(category_label_summaries, ignore_index=True)


## Pair-Level Variation

This section measures variation among annotations for the same premise-hypothesis pair. It does not require annotator IDs.


In [ ]:
def entropy_from_counts(counts):
    values = np.array(list(counts.values()), dtype=float)
    total = values.sum()
    if total == 0:
        return 0.0
    probs = values / total
    return float(-(probs * np.log2(probs)).sum())


def pair_variation(frame):
    rows = []
    for pair_id, group in frame.groupby("pairID"):
        label_counts = Counter(group["label"])
        category_counts = Counter(cat for cats in group["categories"] for cat in cats)
        rows.append({
            "pairID": pair_id,
            "n_annotations": len(group),
            "n_labels": len(label_counts),
            "majority_label": label_counts.most_common(1)[0][0],
            "label_entropy": entropy_from_counts(label_counts),
            "n_categories": len(category_counts),
            "majority_category": category_counts.most_common(1)[0][0],
            "category_entropy": entropy_from_counts(category_counts),
        })
    return pd.DataFrame(rows)


variation_tables = []
for name, frame in datasets.items():
    table = pair_variation(frame)
    table.insert(0, "dataset", name)
    table.to_csv(OUTPUT_DIR / f"{name.lower()}_pair_variation.csv", index=False)
    variation_tables.append(table)

variation = pd.concat(variation_tables, ignore_index=True)
variation_summary = variation.groupby("dataset").agg(
    pairs=("pairID", "count"),
    mean_annotations_per_pair=("n_annotations", "mean"),
    mean_distinct_labels=("n_labels", "mean"),
    mean_label_entropy=("label_entropy", "mean"),
    mean_distinct_categories=("n_categories", "mean"),
    mean_category_entropy=("category_entropy", "mean"),
).round(3).reset_index()
variation_summary.to_csv(OUTPUT_DIR / "pair_variation_summary.csv", index=False)
variation_summary


## Annotator-Level Analysis

This section writes per-annotator label and explanation-category summaries. For readability, LiveNLI plots show the most active workers by default; the CSV files contain all workers.


In [ ]:
def expand_annotators(frame):
    rows = []
    for _, row in frame.iterrows():
        for annotator_id in row["annotator_ids"]:
            rows.append({
                "pairID": row["pairID"],
                "annotator_id": annotator_id,
                "label": row["label"],
                "categories": row["categories"],
                "explanation_words": word_count(row["explanation"]),
            })
    return pd.DataFrame(rows)


def sort_annotator_ids(ids):
    def key(value):
        text = str(value)
        digits = "".join(ch for ch in text if ch.isdigit())
        prefix = text.rstrip(digits)
        return (prefix, int(digits) if digits else -1, text)
    return sorted(ids, key=key)


def annotator_label_stats(expanded):
    counts = expanded.groupby(["annotator_id", "label"]).size().unstack(fill_value=0)
    counts = counts.reindex(columns=LABEL_ORDER, fill_value=0)
    proportions = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    return counts, proportions


def annotator_category_stats(expanded):
    category_expanded = expanded.explode("categories").rename(columns={"categories": "category"})
    counts = category_expanded.groupby(["annotator_id", "category"]).size().unstack(fill_value=0)
    categories = ordered_categories(counts.columns)
    counts = counts.reindex(columns=categories, fill_value=0)
    proportions = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    return counts, proportions


def plot_annotator_stacked(proportions, dataset_name, metric_name, outfile_base, max_annotators=None, activity=None):
    plot_ids = sort_annotator_ids(proportions.index)
    if max_annotators is not None and len(plot_ids) > max_annotators:
        activity = activity if activity is not None else proportions.sum(axis=1)
        plot_ids = sort_annotator_ids(activity.sort_values(ascending=False).head(max_annotators).index)
    plot_data = proportions.loc[plot_ids]
    colors = [LABEL_COLORS.get(col, plt.cm.tab20(i % 20)) for i, col in enumerate(plot_data.columns)]
    ax = plot_data.plot(kind="barh", stacked=True, figsize=(10, max(4, len(plot_data) * 0.35)), color=colors, edgecolor="black")
    ax.set_title(dataset_name)
    ax.set_xlabel(metric_name)
    ax.set_ylabel("Annotator ID")
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.0%}"))
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=min(4, len(plot_data.columns)), frameon=False)
    ax.grid(axis="x", linestyle="--", linewidth=0.5, alpha=0.5)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{outfile_base}.png", dpi=300, bbox_inches="tight")
    plt.savefig(OUTPUT_DIR / f"{outfile_base}.pdf", bbox_inches="tight")
    plt.show()


for name, frame in datasets.items():
    expanded = expand_annotators(frame)
    if expanded.empty:
        raise ValueError(f"{name}: no annotator identifiers found.")

    label_counts, label_props = annotator_label_stats(expanded)
    category_counts, category_props = annotator_category_stats(expanded)

    label_counts.to_csv(OUTPUT_DIR / f"{name.lower()}_annotator_label_counts.csv")
    label_props.to_csv(OUTPUT_DIR / f"{name.lower()}_annotator_label_proportions.csv", float_format="%.4f")
    category_counts.to_csv(OUTPUT_DIR / f"{name.lower()}_annotator_category_counts.csv")
    category_props.to_csv(OUTPUT_DIR / f"{name.lower()}_annotator_category_proportions.csv", float_format="%.4f")

    max_workers = 20 if name == "LiveNLI" else None
    annotator_activity = label_counts.sum(axis=1)
    plot_annotator_stacked(label_props, name, "Label proportion", f"{name.lower()}_annotator_label_distribution", max_annotators=max_workers, activity=annotator_activity)
    plot_annotator_stacked(category_props, name, "Explanation-category proportion", f"{name.lower()}_annotator_category_distribution", max_annotators=max_workers, activity=annotator_activity)

    print(f"{name}: wrote annotator-level tables and figures for {expanded['annotator_id'].nunique()} annotators.")


## Files Written


In [ ]:
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.relative_to(ROOT))
